# LangGraph Memory, Step by Step

You're new to agentic development, so this notebook builds up **one idea at a time**:
every code cell is preceded by a short explanation of *why* it exists, not just what it does.

**The core problem this notebook solves:** an LLM call is stateless — call it twice and it has
no idea the first call ever happened. "Memory" in LangGraph means deliberately capturing state
after each step and handing the right slice of it back on the next call. There are two flavors:

- **Short-term memory** — remembering *this conversation* (a `thread_id`), via a **checkpointer**.
- **Long-term memory** — remembering facts *across* conversations (e.g. "the user's name is Surendra"
  even in a brand new thread), via a **store**.

We'll build both, from the ground up, using nothing but a single plain node — no tools, no ReAct loop,
no multi-agent routing. (Those live in `LangGraph.ipynb` and `langgraph_advanced.ipynb` once you're
ready for them — this notebook is memory and only memory.)

**Roadmap**
1. A stateless graph (and proof that it forgets)
2. Graph state for chat: the `messages` list and why it needs a *reducer*
3. Adding a checkpointer: memory within one conversation
4. Looking under the hood: what a checkpointer actually stores
5. Multiple threads = multiple independent conversations
6. Memory has a cost: trimming a conversation that's grown too long
7. Long-term memory: remembering facts *across* threads with a `Store`
8. Recap


## Setup

Loading the API key from `.env` and creating one shared `ChatOpenAI` instance that every
section below reuses. Nothing memory-specific yet — just getting a model we can call.


In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")


## 1. A Stateless Graph (and proof that it forgets)

First, the smallest possible chat graph: one state shape holding a single `question` and `answer`,
one node that calls the LLM, wired `START -> chat -> END`. No memory involved at all — this is our
baseline, so the next sections have something concrete to fix.


In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    question: str
    answer: str


def chat_node(state: State) -> dict:
    response = llm.invoke(state["question"])
    return {"answer": response.content}


graph = StateGraph(State)
graph.add_node("chat", chat_node)
graph.add_edge(START, "chat")
graph.add_edge("chat", END)

app = graph.compile()


Two `.invoke()` calls, the second one referring back to the first. Watch what happens —
there is nothing wired up yet to make the second call aware of the first.


In [3]:
first = app.invoke({"question": "My name is Surendra. Remember that."})
print("Q1:", first["question"])
print("A1:", first["answer"])

second = app.invoke({"question": "What is my name?"})
print()
print("Q2:", second["question"])
print("A2:", second["answer"])


Q1: My name is Surendra. Remember that.
A1: I can remember your name for this conversation, Surendra. However, I won't be able to recall it in future chats. How can I assist you today?



Q2: What is my name?
A2: I'm sorry, but I don't know your name. If you'd like to share it, feel free to do so!


The model has no idea what your name is in the second call — `app.invoke()` only ever sees
the `State` dict you pass it *that call*. There is no hidden history. Each invocation starts from
a blank slate, which is exactly what "stateless" means here. Fixing this is the rest of the notebook.


## 2. Graph State for Chat: `messages` and Reducers

To remember a conversation we need state that can **grow** — a list of messages — rather than state
that gets overwritten each turn. By default, when a node returns `{"key": value}`, LangGraph
*replaces* `state["key"]` with `value`. That's wrong for a message list: we want to **append**,
not replace.

The fix is a **reducer**: an annotation on the state field that tells LangGraph how to combine the
old value with the new one. `add_messages` is the built-in reducer for chat history — it appends new
messages (and can also update/delete existing ones by message `id`, which is how tool-call results
get merged in later, more advanced notebooks).


In [4]:
from typing import Annotated
from langchain_core.messages import HumanMessage
from langgraph.graph.message import add_messages


class ChatState(TypedDict):
    messages: Annotated[list, add_messages]


def chat_with_history(state: ChatState) -> dict:
    response = llm.invoke(state["messages"])
    # Returning a list here means "append these to messages", thanks to add_messages.
    return {"messages": [response]}


chat_graph = StateGraph(ChatState)
chat_graph.add_node("chat", chat_with_history)
chat_graph.add_edge(START, "chat")
chat_graph.add_edge("chat", END)

chat_app = chat_graph.compile()


Still no checkpointer, so this graph is *still* stateless between calls — but now we're passing
the **whole message list** back in ourselves each time, which is the manual version of what a
checkpointer will soon automate.


In [5]:
result = chat_app.invoke({"messages": [HumanMessage(content="My name is Surendra.")]})
history = result["messages"]
for m in history:
    m.pretty_print()

print("\n--- turn 2, manually re-sending history ---\n")

history.append(HumanMessage(content="What is my name?"))
result = chat_app.invoke({"messages": history})
for m in result["messages"]:
    m.pretty_print()


================================ Human Message =================================

My name is Surendra.
================================== Ai Message ==================================

Nice to meet you, Surendra! How can I assist you today?

--- turn 2, manually re-sending history ---



================================ Human Message =================================

My name is Surendra.
================================== Ai Message ==================================

Nice to meet you, Surendra! How can I assist you today?
================================ Human Message =================================

What is my name?
================================== Ai Message ==================================

Your name is Surendra. How can I help you further?


That worked — but only because *we* remembered to carry `history` forward by hand. In a real
app you don't want every caller responsible for stashing and resending the full transcript. That
bookkeeping is exactly what a checkpointer takes over next.


## 3. Adding a Checkpointer: Memory Within One Conversation

A **checkpointer** saves the graph's state after every step, keyed by a `thread_id` you pass in
`config`. Call the graph again with the *same* `thread_id` and LangGraph loads the saved state
before running your node — so you only ever send the **new** message, never the whole history.

`InMemorySaver` keeps checkpoints in a plain Python dict in this process — perfect for learning and
local dev, gone the moment the kernel restarts. (Production swaps in a durable backend, e.g.
`langgraph-checkpoint-sqlite` or `langgraph-checkpoint-postgres`, behind the exact same interface —
you'll see the difference is a one-line swap in the recap.)


In [6]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

# Compiling with a checkpointer is the only change from the graph above.
memory_app = chat_graph.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "conversation-1"}}


Now repeat the exact same two-turn conversation as above — but this time we hand the graph
only the *new* `HumanMessage` each call, never the accumulated history ourselves.


In [7]:
result = memory_app.invoke(
    {"messages": [HumanMessage(content="My name is Surendra.")]},
    config=config,
)
result["messages"][-1].pretty_print()

result = memory_app.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config=config,
)
result["messages"][-1].pretty_print()


================================== Ai Message ==================================

Nice to meet you, Surendra! How can I assist you today?


================================== Ai Message ==================================

Your name is Surendra!


It remembered — without us ever resending history. The `thread_id` is doing all the work:
it tells the checkpointer *which* saved conversation to load before this call, and to save the
updated one back after.


## 4. Under the Hood: What a Checkpointer Actually Stores

"Memory" can feel magical from the outside. It isn't — `app.get_state(config)` lets you inspect
exactly what's saved for a `thread_id` right now, and `app.get_state_history(config)` shows every
checkpoint ever written for it, oldest state first is last. This is the same mechanism that powers
features like "rewind the conversation to 3 turns ago" or "resume after a crash".


In [8]:
snapshot = memory_app.get_state(config)

print("Number of saved messages:", len(snapshot.values["messages"]))
for m in snapshot.values["messages"]:
    print(f"  {m.type}: {m.content}")

print("\nnext node to run (empty means the graph finished):", snapshot.next)


Number of saved messages: 4
  human: My name is Surendra.
  ai: Nice to meet you, Surendra! How can I assist you today?
  human: What is my name?
  ai: Your name is Surendra!

next node to run (empty means the graph finished): ()


In [9]:
print(f"{'step':>4}  {'# messages':>10}   latest message")
for i, checkpoint in enumerate(memory_app.get_state_history(config)):
    msgs = checkpoint.values.get("messages", [])
    latest = msgs[-1].content if msgs else "(empty)"
    print(f"{i:>4}  {len(msgs):>10}   {latest[:60]}")


step  # messages   latest message
   0           4   Your name is Surendra!
   1           3   What is my name?
   2           2   Nice to meet you, Surendra! How can I assist you today?
   3           2   Nice to meet you, Surendra! How can I assist you today?
   4           1   My name is Surendra.
   5           0   (empty)


Each row is a full snapshot taken after one graph step. This is *literally* what "memory"
is under the hood — no separate database schema to design, no manual serialization code to write.


## 5. Multiple Threads = Multiple Independent Conversations

`thread_id` is just a string key you choose — one per independent conversation, e.g. one per chat
session, one per user, one per support ticket. A different `thread_id` on the *same compiled graph*
starts from a completely blank state.


In [10]:
other_config = {"configurable": {"thread_id": "conversation-2"}}

result = memory_app.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config=other_config,
)
result["messages"][-1].pretty_print()


================================== Ai Message ==================================

I'm sorry, but I don't have access to personal information about individuals unless it has been shared with me in the course of our conversation. If you'd like, you can tell me your name!


`conversation-2` has never heard of Surendra — it's a fresh thread. Meanwhile `conversation-1`
is untouched and still remembers, because the two threads' state lives at separate keys.


In [11]:
result = memory_app.invoke(
    {"messages": [HumanMessage(content="Say my name one more time.")]},
    config=config,  # back to conversation-1
)
result["messages"][-1].pretty_print()


================================== Ai Message ==================================

Sure! Your name is Surendra.


## 6. Memory Has a Cost: Trimming Long Conversations

Every saved message gets resent to the LLM on every future turn in that thread — a checkpointer
remembers *everything*, forever, by default. That's correct behavior, but it means cost and
latency both grow linearly with conversation length, and eventually you'll hit the model's context
window. `trim_messages` lets a node cap how much history actually reaches the LLM, independent of
how much the checkpointer has saved (the checkpointer still keeps the full, untrimmed log).

Below, `max_tokens=60` is deliberately tiny so the trimming is visible in a short demo — in a real
app you'd size it to a large fraction of the model's context window.


In [12]:
from langchain_core.messages import trim_messages


def chat_with_trimming(state: ChatState) -> dict:
    trimmed = trim_messages(
        state["messages"],
        max_tokens=60,
        token_counter=llm,
        strategy="last",     # keep the most recent messages that fit
        start_on="human",    # don't cut the window off mid-exchange
    )
    response = llm.invoke(trimmed)
    return {"messages": [response]}


trimming_graph = StateGraph(ChatState)
trimming_graph.add_node("chat", chat_with_trimming)
trimming_graph.add_edge(START, "chat")
trimming_graph.add_edge("chat", END)

trimming_app = trimming_graph.compile(checkpointer=InMemorySaver())
trim_config = {"configurable": {"thread_id": "long-conversation"}}


In [13]:
topics = [
    "My favorite color is blue.",
    "My favorite food is pasta.",
    "My favorite city is Tokyo.",
    "My favorite animal is the fox.",
    "What is my favorite color? Answer in one short sentence.",
]

for topic in topics:
    result = trimming_app.invoke(
        {"messages": [HumanMessage(content=topic)]},
        config=trim_config,
    )

result["messages"][-1].pretty_print()

full_history = trimming_app.get_state(trim_config).values["messages"]
print(f"\ncheckpointer kept all {len(full_history)} messages, "
      f"but the final LLM call only saw the trimmed tail.")


================================== Ai Message ==================================

I don't know your favorite color.

checkpointer kept all 10 messages, but the final LLM call only saw the trimmed tail.


Notice the model says it doesn't know the favorite color — "blue" fell outside the 60-token trimming window by the last turn, even though the checkpointer still has it saved in full (check `full_history` above). This is the real tradeoff: trimming controls *cost*, not *forgetting* — the data survives in the checkpoint even when the model can no longer see it. A common middle ground, not implemented here, is a summarization node that periodically compresses old turns into one summary message instead of dropping them outright.

## 7. Long-Term Memory: Facts That Survive Across Threads

A checkpointer's memory is scoped to one `thread_id` — by design, `conversation-2` above couldn't
see anything from `conversation-1`. But some facts (a user's name, their preferences) should be
recalled in *any* future conversation with that user, not just the thread they were mentioned in.
That's what a **`Store`** is for: a separate key-value memory, namespaced however you like (e.g. by
`user_id`), that every thread can read from and write to.

We'll namespace by `("memories", user_id)` and store simple facts under string keys.


In [14]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

USER_ID = "surendra"
NAMESPACE = ("memories", USER_ID)

store.put(NAMESPACE, "name", {"fact": "The user's name is Surendra."})
store.put(NAMESPACE, "role", {"fact": "The user is new to agentic development."})

for item in store.search(NAMESPACE):
    print(item.key, "->", item.value)


name -> {'fact': "The user's name is Surendra."}
role -> {'fact': 'The user is new to agentic development.'}


To use this inside a graph, a node needs access to the store. LangGraph injects it
automatically when you compile *with* a store and declare a `store` parameter on the node function —
no need to thread it through `State` yourself. Here the node reads every fact under the user's
namespace and stuffs it into the system prompt before calling the LLM.


In [15]:
from langchain_core.messages import SystemMessage
from langgraph.store.base import BaseStore


def chat_with_long_term_memory(state: ChatState, *, store: BaseStore) -> dict:
    facts = store.search(NAMESPACE)
    facts_text = "\n".join(f"- {item.value['fact']}" for item in facts)

    system = SystemMessage(content=f"Known facts about this user:\n{facts_text}")
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}


long_term_graph = StateGraph(ChatState)
long_term_graph.add_node("chat", chat_with_long_term_memory)
long_term_graph.add_edge(START, "chat")
long_term_graph.add_edge("chat", END)

# Two separate memories, both wired up: checkpointer for short-term (this thread's messages),
# store for long-term (facts shared across every thread).
long_term_app = long_term_graph.compile(checkpointer=InMemorySaver(), store=store)


Now ask a **brand-new thread** — one that has never seen a single message before — who the
user is. It has no checkpointed history to draw on, only the store.


In [16]:
brand_new_thread = {"configurable": {"thread_id": "conversation-never-seen-before"}}

result = long_term_app.invoke(
    {"messages": [HumanMessage(content="What do you know about me?")]},
    config=brand_new_thread,
)
result["messages"][-1].pretty_print()


================================== Ai Message ==================================

I know that your name is Surendra and that you are new to agentic development. If there's anything specific you'd like to share or discuss, feel free to let me know!


It knew, despite this thread never having exchanged a message before — because that knowledge
lives in the `Store`, not in any one thread's checkpoint. This is the pattern real assistants use for
"remembering you" across sessions: a checkpointer for *this conversation's* flow of messages, a store
for *durable facts about the user*, populated either by hand (as above) or by a node that extracts
and saves facts as the conversation happens.


## Recap

| Concept | What it solves | Key API |
|---|---|---|
| `Annotated[list, add_messages]` | State that appends instead of overwrites | `langgraph.graph.message.add_messages` |
| Checkpointer | Remembering *this* conversation | `InMemorySaver`, `.compile(checkpointer=...)`, `thread_id` |
| `get_state` / `get_state_history` | Inspecting/debugging what's saved | `app.get_state(config)` |
| Trimming | Bounding cost/latency as a thread grows | `trim_messages(...)` inside a node |
| Store | Facts that outlive a single thread | `InMemoryStore`, `.compile(store=...)`, a `store` node argument |

**Swapping in durable memory:** everything here used in-memory backends that vanish when the kernel
restarts. Moving to production is a one-line change in what you pass to `.compile()` — e.g.
`from langgraph.checkpoint.sqlite import SqliteSaver` (needs `uv add langgraph-checkpoint-sqlite`) —
the rest of every graph above is unchanged, because checkpointers and stores are interchangeable
behind the same interface.

**Where to go next:**
- `LangGraph.ipynb` — the ReAct tool-calling pattern (a graph that decides *when* to call tools)
- `langgraph_advanced.ipynb` — `create_agent`, human-in-the-loop approval gates, and multi-agent
  supervisors — all of which combine with the memory patterns you just built here.
